# Notebook 4 — Incorporating Text Reports (Multimodal Learning)
### AIMI High School Internship

A chest X-ray rarely travels alone — it comes with a **radiologist's report** in plain English. Can a model that reads *both* the image **and** the text beat one that sees only pixels?

A new dataset (**Dataset 2**): chest X-rays where each study also has a real **radiology report** (`findings` / `impression`) and a `pneumonia` label. This is different from the Dataset 1 data — here the text is the star.

Turn the reports into numbers with **ClinicalBERT** (a language model trained on medical text), fuse those with image features from a **ResNet-50**, and race the multimodal model against a vision-only baseline. Then we let an AI (**MedGemma**) *write* a report from an image and compare it to the real one.

## What you'll do in this notebook

1. **Setup** — load Dataset 2 images + reports.
2. **The reports we already have** — read real radiology text and see how it relates to the label.
3. **Text → numbers with ClinicalBERT** — turn each report into an embedding vector.
4. **A multimodal model** — fuse image features + text features (late fusion).
5. **The race: vision-only vs multimodal** — train both, plot accuracy, and discuss *why*.
6. **Bonus — an AI radiologist (MedGemma)** — generate a report from an image and compare it to the real one.
7. **Side quest — supplement Dataset 1 with Dataset 2** — reuse what you learn here to make your Notebook 2 / 3 models better.

> **Side goal of this notebook:** Dataset 2 has something Dataset 1 doesn't — *text reports*, and a lot more chest X-rays. Keep asking yourself: *how could this richer data make my pneumonia classifier (Notebook 2) or detector (Notebook 3) better?* Section 7 gives you concrete ways to try it.

# 1. Setup

> Before running: **Runtime → Change runtime type → T4 GPU**, and make sure `dataset_2.zip` is in your Drive ("My Drive").

In [ ]:
#@title Notebook settings { display-mode: "form" }
# (Automated validation overrides DATA_DIR via papermill — do not rename this cell.)
from pathlib import Path
# Put `dataset_2.zip` in your Google Drive ("My Drive"), then run all cells.
DATA_DIR = Path("/content/drive/MyDrive/dataset_2")
EPOCHS  = 3    #@param {type:"integer"}

# How many images to hand MedGemma in the bonus section (generation is slow on a T4).
N_MEDGEMMA = 8  #@param {type:"integer"}

In [ ]:
#@title Run setup (imports · data · device) { display-mode: "form" }
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# Mount Google Drive so we can read the dataset (skipped automatically off Colab).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# Unzip the dataset to Colab's local disk and read from there. Reading images over the Drive
# mount re-fetches each file from the network on every read (slow); we unzip the dataset's
# .zip once to /content instead. Students put dataset_1.zip / dataset_2.zip in their Drive
# (My Drive). Skipped when DATA_DIR already points at local data (validation / off-Colab).
import zipfile
DATA_DIR = Path(DATA_DIR)
if str(DATA_DIR).startswith("/content/drive/"):
    local = Path("/content") / DATA_DIR.name
    if not local.exists():
        zip_path = Path("/content/drive/MyDrive") / f"{DATA_DIR.name}.zip"
        assert zip_path.exists(), (
            f"{zip_path} not found — add {DATA_DIR.name}.zip to your Google Drive (My Drive).")
        print(f"Unzipping {DATA_DIR.name}.zip to local disk (one-time)...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(local)                            # everything lands under /content/<name>
        kids = [p for p in local.iterdir() if not p.name.startswith((".", "__"))]
        if len(kids) == 1 and kids[0].is_dir():            # zip wrapped files in one extra folder
            import shutil
            for it in list(kids[0].iterdir()):
                shutil.move(str(it), str(local / it.name))
            kids[0].rmdir()
    DATA_DIR = local
assert DATA_DIR.exists(), f"dataset not found at {DATA_DIR}."
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEXT_COL = "impression"      # the real radiology text we'll use (always present)

def load_split(name, n=None):
    """Load a labels CSV, keep usable rows, optionally subsample."""
    df = pd.read_csv(DATA_DIR / f"{name}_labels.csv")
    df = df[df["pneumonia"].isin([0, 1])].copy()                 # drop the -1 "uncertain" label
    df["img_path"] = df["path_to_image"].apply(lambda p: DATA_DIR / p)
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
    df = df[df[TEXT_COL].str.strip() != ""]
    df = df.drop_duplicates("study_id").reset_index(drop=True)   # one image per study
    if n:
        df = df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)
    return df

train_df = load_split("train")
val_df   = load_split("val")

# Quick sanity check on ONE image. (Every label row in this dataset has its image, so
# checking all of them per-row would be thousands of slow stat() calls — one probe is enough.)
probe = train_df["img_path"].iloc[0]
assert probe.exists(), (
    f"\nImage not found: {probe}\n"
    f"The CSVs loaded but the images aren't where DATA_DIR points — check that the dataset_2 "
    f"folder in your Drive contains the images/ subfolder, then re-run this cell.")

print(f"device={device} | train={len(train_df)} | val={len(val_df)} | text column='{TEXT_COL}'")
print("pneumonia balance (train):", train_df["pneumonia"].value_counts().to_dict())

# 2. The reports we already have

Unlike Dataset 1, every study here carries a real **radiology report**. Read a few — notice they're written in dense clinical shorthand, and that the **impression** often names the key finding.

In [ ]:
#@title An X-ray and its real report { display-mode: "form" }
row = train_df.iloc[0]
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(Image.open(row["img_path"]).convert("L"), cmap="gray")
ax.set_title(f"pneumonia label = {row['pneumonia']}"); ax.axis("off"); plt.show()
print("IMPRESSION:\n", row["impression"][:600])

The impression frequently *states the finding* ("…consistent with pneumonia"). That makes the text extremely informative — maybe *too* informative. Keep this in mind: if the report already contains the answer, is using it "fair"? We'll come back to this in Section 5. (When would a model realistically have the report? When a radiologist has already dictated it and we want a second-reader model.)

# 3. Text → numbers with ClinicalBERT

A neural network needs numbers, not words. **ClinicalBERT** reads a report and outputs a 768-dimensional **embedding** — a vector that captures its meaning. We run it once over every report (no training — the model is already trained) and cache the results.

The convention: BERT prepends a special `[CLS]` token to every input; the model's hidden state *at that token* is used as the whole-sentence summary. Pulling it out is your task.

In [ ]:
bert_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert = AutoModel.from_pretrained(bert_name).to(device).eval()   # eval = frozen, no learning

@torch.no_grad()
def embed_reports(texts, batch_size=32):
    """Turn a list of report strings into a [N, 768] array of embeddings."""
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="  embedding", leave=False):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True,
                           padding=True, max_length=128).to(device)
        hidden = bert(**inputs).last_hidden_state          # [batch, tokens, 768]
        # Take the [CLS] token's vector (position 0) as the sentence embedding.
        # TODO: select the [CLS] token embedding from `hidden`.
        # hidden has shape [batch, tokens, 768]; [CLS] is token index 0.
        # -> cls = hidden[:, 0, :]
        cls = ...  # YOUR CODE HERE
        out.append(cls.cpu().numpy())
    return np.concatenate(out, axis=0)

train_text = embed_reports(train_df["impression"].tolist())
val_text   = embed_reports(val_df["impression"].tolist())
print("text embeddings:", train_text.shape, "(N reports × 768)")

# 4. A multimodal model (late fusion)

**Late fusion** = let each modality form its own feature vector, then **concatenate** them and classify the combined vector:

- **Image branch** — ResNet-50 → 2048 numbers.
- **Text branch** — the 768-dim ClinicalBERT embedding → squeezed to 256 numbers.
- **Fusion head** — concatenate (2048 + 256 = 2304) → a final pneumonia score.

In [ ]:
class LateFusionModel(nn.Module):
    def __init__(self, text_dim=768):
        super().__init__()
        self.vision = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.vision.fc = nn.Identity()                    # keep the 2048-dim features
        self.text_fc = nn.Sequential(nn.Linear(text_dim, 256), nn.ReLU(), nn.Dropout(0.3))
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 256, 512), nn.ReLU(), nn.Dropout(0.4), nn.Linear(512, 1))

    def forward(self, images, text_emb):
        img_feat = self.vision(images)                    # [batch, 2048]
        txt_feat = self.text_fc(text_emb)                 # [batch, 256]
        # Combine the two modalities into one vector, then classify.
        # TODO: concatenate img_feat and txt_feat along dim=1 (the feature axis).
        # -> fused = torch.cat([...], dim=1)
        fused = ...  # YOUR CODE HERE
        return self.classifier(fused)

# A vision-only control that IGNORES the text — our baseline to beat.
class VisionOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.vision.fc = nn.Linear(2048, 1)
    def forward(self, images, text_emb):                  # same signature; text ignored
        return self.vision(images)

print("models defined.")

The dataset returns three things per study: the **image**, its **precomputed text embedding**, and the **label**.

In [ ]:
IMAGENET = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class MultimodalDataset(Dataset):
    def __init__(self, df, text_emb, transform=IMAGENET):
        self.df, self.text_emb, self.transform = df.reset_index(drop=True), text_emb, transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.transform(Image.open(row["img_path"]).convert("RGB"))
        txt = torch.tensor(self.text_emb[idx], dtype=torch.float32)
        label = torch.tensor([float(row["pneumonia"])])
        return img, txt, label

train_loader = DataLoader(MultimodalDataset(train_df, train_text), batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(MultimodalDataset(val_df,   val_text),   batch_size=32, shuffle=False, num_workers=2)
print("loaders ready.")

# 5. The race — does text help?

In science you compare against a control. We train **both** models the same way and compare validation accuracy each epoch. If text adds signal, the multimodal model should win.

In [ ]:
def calculate_accuracy(outputs, labels):
    preds = (torch.sigmoid(outputs) >= 0.5).float()
    return (preds == labels).float().mean().item()

def train_and_evaluate(model, epochs=EPOCHS):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    history = []
    for epoch in range(epochs):
        model.train()
        for images, texts, labels in tqdm(train_loader, desc=f"  epoch {epoch+1} train", leave=False):
            images, texts, labels = images.to(device), texts.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images, texts), labels)
            loss.backward()
            optimizer.step()
        model.eval()
        accs = []
        with torch.no_grad():
            for images, texts, labels in val_loader:
                images, texts, labels = images.to(device), texts.to(device), labels.to(device)
                accs.append(calculate_accuracy(model(images, texts), labels))
        history.append(float(np.mean(accs)))
        print(f"  epoch {epoch+1} | val acc {history[-1]*100:.1f}%")
    return history

print(" Vision only:")
vision_acc = train_and_evaluate(VisionOnlyModel())
print("\n Vision + Text (multimodal):")
fusion_acc = train_and_evaluate(LateFusionModel())

In [ ]:
#@title Vision-only vs multimodal { display-mode: "form" }
ep = range(1, len(vision_acc) + 1)
plt.figure(figsize=(8, 5))
plt.plot(ep, [a*100 for a in vision_acc], "o-", color="#4C72B0", label="Vision only")
plt.plot(ep, [a*100 for a in fusion_acc], "o-", color="#DD8452", label="Vision + Text")
plt.title("Does adding the report help?"); plt.xlabel("epoch"); plt.ylabel("val accuracy (%)")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print(f"final: vision-only {vision_acc[-1]*100:.1f}%  |  multimodal {fusion_acc[-1]*100:.1f}%")

Where did the signal come from? If the multimodal model beat vision-only, remember Section 2: the impression often *states* the diagnosis. The model may be "reading the answer key," not interpreting the X-ray. That's not cheating *if* a radiologist's report is genuinely available at prediction time (e.g. a second-reader tool) — but it would be misleading to claim the model "diagnoses from images."

Try it: test the leakage hypothesis. Re-run using `findings` instead of `impression` (often less conclusive), or use the **MedGemma-generated** reports from Section 6 (which never saw the label). Does the boost survive?

# 6. Bonus — an AI radiologist (MedGemma)

What if a report *didn't* exist? Could an AI write one? **MedGemma** is a **Vision-Language Model**: it looks at an image and writes text. We'll load it (compressed to 4-bit so it fits a T4), have it describe a few X-rays, and compare to the real report.

> **Note:** MedGemma is a **gated** model on Hugging Face — accept its license at the model page and log in with `huggingface_hub` once. Generation is slow on a T4 (~10–20 s/image), so we only do `N_MEDGEMMA` images.

In [ ]:
#@title Load MedGemma (4-bit) { display-mode: "form" }
from transformers import pipeline, BitsAndBytesConfig
MEDGEMMA_ID = "google/medgemma-1.5-4b-it"      # gated; accept the license on HF first
try:
    medgemma = pipeline(
        "image-text-to-text", model=MEDGEMMA_ID,
        model_kwargs={"torch_dtype": torch.bfloat16, "device_map": "auto",
                      "quantization_config": BitsAndBytesConfig(load_in_4bit=True)})
    MEDGEMMA_OK = True
    print("MedGemma loaded.")
except Exception as e:
    MEDGEMMA_OK = False
    print("Could not load MedGemma (gated access / memory?). Skipping live generation.\n", str(e)[:200])

In [ ]:
def medgemma_report(img_path):
    """Ask MedGemma to write a radiology report for one chest X-ray."""
    image = Image.open(img_path).convert("RGB")
    # Give the model a radiologist PERSONA and a clear TASK.
    # TODO: write two prompt strings that steer MedGemma:
    # system = a one-sentence PERSONA ("You are an expert radiologist ...")
    # task   = a one-sentence INSTRUCTION ("Write a concise radiology report ...")
    system = ...  # YOUR CODE HERE
    task   = ...  # YOUR CODE HERE
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system}]},
        {"role": "user", "content": [{"type": "text", "text": task},
                                     {"type": "image", "image": image}]}]
    out = medgemma(text=messages, max_new_tokens=200, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

if MEDGEMMA_OK:
    sub = train_df.head(N_MEDGEMMA)
    for _, r in sub.iterrows():
        print("=" * 70)
        print("AI (MedGemma) report:\n", medgemma_report(r["img_path"]).strip()[:500])
        print("\nReal radiologist impression:\n", r["impression"].strip()[:300])
else:
    print("MedGemma not available — skipping. The rest of the notebook does not depend on it.")

**Compare them.** Where does the AI report agree with the radiologist? Where does it hallucinate or hedge?

**Try it:** generate MedGemma reports for, say, 100 training images, embed *those* with ClinicalBERT, and re-run the race in Section 5. Because MedGemma never saw the label, any boost it gives is *real* added signal from the image — not leakage. Does multimodal still win?

# 7. Side quest — use Dataset 2 to strengthen your Dataset 1 models

Step back and notice what you have. **Dataset 1** (Notebooks 2, 3, 5) is a few thousand chest X-rays with a pneumonia label — and *no text*. **Dataset 2** (this notebook) has chest X-rays **with radiology reports**. Two questions worth a real experiment:

> Can the *extra images* and the *report supervision* in Dataset 2 make my Dataset 1 classifier or detector **better** — especially since Dataset 1 is small?

This is exactly how real medical-AI teams squeeze more out of limited labeled data: **pretrain on a larger, related dataset, then fine-tune on the target task.** Below is a starter for the most direct version — exporting an image encoder that has already "seen" a lot of chest X-rays — plus a menu of other directions to try.

In [ ]:
#@title Export a Dataset-2-pretrained image encoder for Notebook 2 / 3 { display-mode: "form" }
# The race in Section 5 trained models and threw them away. Here we train ONE multimodal
# model and KEEP it, then save just its ResNet-50 vision branch. That backbone has now
# seen far more chest X-rays than Dataset 1 has — and learned features shaped by the reports.
encoder_model = LateFusionModel()
train_and_evaluate(encoder_model)            # trains in place on Dataset 2 (image + report)

# Save to Drive so Notebook 2 / 3 can load it later (a local /content file won't survive
# the jump to another notebook / runtime).
ENCODER_PATH = "/content/drive/MyDrive/chexpert_resnet50_encoder.pth"
try:
    torch.save(encoder_model.vision.state_dict(), ENCODER_PATH)
    print("saved Dataset-2-pretrained ResNet-50 encoder ->", ENCODER_PATH)
except Exception:
    ENCODER_PATH = "chexpert_resnet50_encoder.pth"
    torch.save(encoder_model.vision.state_dict(), ENCODER_PATH)
    print("Drive not available; saved locally ->", ENCODER_PATH)

## 7.1 Use it in Notebook 2 (or 3)

Notebook 2's `make_model` already wires up a `resnet50` backbone, so warm-start it with the encoder you just trained, then fine-tune on Dataset 1 as usual:

```python
model = make_model("resnet50")                  # NB2 helper (ImageNet-pretrained to start)
enc = torch.load("/content/drive/MyDrive/chexpert_resnet50_encoder.pth")
model.load_state_dict(enc, strict=False)        # load the chest-X-ray backbone; keep NB2's fresh head
```

`strict=False` loads the backbone weights while leaving Notebook 2's classification head untouched. Train as before and compare val AUC to the plain ImageNet baseline — **does starting from chest X-rays beat starting from ImageNet on the smaller Dataset 1?** (Use `resnet50` in NB2 so the weights line up.)

## 7.2 More ways to let Dataset 2 help Dataset 1 — pick one and experiment

- **Pool the data.** Both datasets carry a binary pneumonia label. Train a single classifier on Dataset 1 **+** Dataset 2 images and see if the bigger, more varied training set generalizes better. Watch for **domain shift** (different hospitals, scanners, populations) — the exact confounding you studied in Notebooks 1 and 5.
- **Cross-dataset stress test.** Train on Dataset 2, evaluate on Dataset 1 (or the reverse). A score drop *is* the domain gap — an honest measure of whether your model learned pneumonia or learned one hospital's quirks.
- **Reports as a teacher.** The reports are free, rich supervision Dataset 1 lacks. Use the ClinicalBERT embeddings (Section 3) or MedGemma (Section 6) to mine extra structure — e.g., pseudo-label Dataset 1 images with a Dataset-2-trained model, or distill report knowledge into an image-only model.

**Your turn:** run the encoder-transfer experiment above, log the AUC change in Notebook 2, and write one sentence on *why* domain-pretraining did (or didn't) help on a dataset this small.